# Phase 3: DeBERTa-v3-base Evaluation on ANLI Round 2

**Objective:** Evaluate a pre-trained DeBERTa-v3-base NLI model on ANLI Round 2, then perform comprehensive error analysis connecting back to Phase 1 EDA findings.

**Model:** `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`
- Fine-tuned on MNLI + Fever-NLI + ANLI (all 3 rounds) — 763,913 NLI pairs
- DeBERTa-v3-base backbone (~86M params) with disentangled attention + RTD pre-training
- Outperforms most *large* models on the ANLI benchmark

**Why this model?** ANLI R2 was created adversarially against RoBERTa. DeBERTa-v3 uses a fundamentally different architecture (disentangled attention) and pre-training objective (replaced token detection), giving it a structural advantage. Training on the combined MNLI + FEVER-NLI + ANLI data is the recipe recommended by the original ANLI paper authors — transfer from diverse NLI data significantly improves performance on adversarial examples.

## Setup and Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

In [4]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


In [6]:
CONFIG = {
    "model_name": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    "num_labels": 3,
    "max_length": 256,   # From Phase 1 EDA
    "batch_size": 32,    # For inference — can be larger since no gradients
}

In [7]:
LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}
LABEL_NAMES = ["entailment", "neutral", "contradiction"]

In [8]:
# Note: MoritzLaurer's model uses label ordering: contradiction=0, neutral=1, entailment=2
# We need to verify and remap to match ANLI's ordering: entailment=0, neutral=1, contradiction=2
MODEL_LABEL_MAP = None  # Will be set after loading the model

In [9]:
print("=== Configuration ===")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

=== Configuration ===
  model_name: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  num_labels: 3
  max_length: 256
  batch_size: 32


## Load Model and Tokenizer
This pre-trained model uses `AutoTokenizer` successfully because MoritzLaurer's model card includes properly configured tokenizer files.
If it fails, we fall back to our Phase 1 approach (explicit `DebertaV2Tokenizer` with `vocab_file`).

In [10]:
from transformers import DebertaV2Tokenizer
from huggingface_hub import hf_hub_download

try:
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    print("Loaded tokenizer via AutoTokenizer")
except Exception as e:
    print(f"AutoTokenizer failed: {e}")
    print("Falling back to explicit DebertaV2Tokenizer...")
    spm_path = hf_hub_download(
        repo_id=CONFIG["model_name"],
        filename="spm.model",
    )
    tokenizer = DebertaV2Tokenizer(vocab_file=spm_path, do_lower_case=False)
    print(f"Loaded tokenizer via DebertaV2Tokenizer")

model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"])
model.to(device)
model.eval()

Loaded tokenizer via AutoTokenizer


Loading weights: 100%|█████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 2295.79it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [11]:
# Check model's label mapping
print(f"\nModel label mapping (id2label): {model.config.id2label}")
print(f"Model label mapping (label2id): {model.config.label2id}")


Model label mapping (id2label): {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
Model label mapping (label2id): {'contradiction': 2, 'entailment': 0, 'neutral': 1}


In [12]:
# Build remapping: model label index -> ANLI label index
# ANLI: entailment=0, neutral=1, contradiction=2
# Model may use different ordering
model_id2label = model.config.id2label
REMAP = {}
for model_idx, label_str in model_id2label.items():
    model_idx = int(model_idx)
    label_lower = label_str.lower()
    if "entail" in label_lower:
        REMAP[model_idx] = 0
    elif "neutral" in label_lower:
        REMAP[model_idx] = 1
    elif "contra" in label_lower:
        REMAP[model_idx] = 2

print(f"Remap (model_idx -> ANLI_idx): {REMAP}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1e9:.2f} GB (fp32)")

Remap (model_idx -> ANLI_idx): {0: 0, 1: 1, 2: 2}

Total parameters: 184,424,451
Model size: ~0.74 GB (fp32)


## Load Data

In [13]:
dataset = load_dataset("facebook/anli")

dev_ds = dataset["dev_r2"]
test_ds = dataset["test_r2"]

print(f"Dev:  {len(dev_ds):,} examples")
print(f"Test: {len(test_ds):,} examples")

Dev:  1,000 examples
Test: 1,000 examples


In [14]:
# Also load as pandas for error analysis later
dev_df = dev_ds.to_pandas()
test_df = test_ds.to_pandas()
for df in [dev_df, test_df]:
    df["label_name"] = df["label"].map(LABEL_MAP)

## Inference

In [15]:
def run_inference(premises, hypotheses, batch_size=32):
    """Run batched inference and return predicted labels + probabilities."""
    all_logits = []

    for i in range(0, len(premises), batch_size):
        batch_premises = premises[i:i+batch_size]
        batch_hypotheses = hypotheses[i:i+batch_size]

        inputs = tokenizer(
            batch_premises,
            batch_hypotheses,
            max_length=CONFIG["max_length"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        all_logits.append(outputs.logits.cpu())

        if (i // batch_size) % 10 == 0:
            print(f"  Processed {min(i+batch_size, len(premises))}/{len(premises)}")

    logits = torch.cat(all_logits, dim=0)
    probs = torch.softmax(logits, dim=-1).numpy()

    # Get predicted class in model's label space, then remap to ANLI labels
    model_preds = np.argmax(logits.numpy(), axis=-1)
    anli_preds = np.array([REMAP[p] for p in model_preds])

    # Remap probabilities to ANLI ordering: [entailment, neutral, contradiction]
    anli_probs = np.zeros_like(probs)
    for model_idx, anli_idx in REMAP.items():
        anli_probs[:, anli_idx] = probs[:, model_idx]

    return anli_preds, anli_probs

In [ ]:
print("Running inference on Dev set...")
dev_preds, dev_probs = run_inference(
    dev_df["premise"].tolist(),
    dev_df["hypothesis"].tolist(),
    batch_size=CONFIG["batch_size"],
)

Running inference on Dev set...


In [ ]:
print("\nRunning inference on Test set...")
test_preds, test_probs = run_inference(
    test_df["premise"].tolist(),
    test_df["hypothesis"].tolist(),
    batch_size=CONFIG["batch_size"],
)

## Evaluation Results

In [ ]:
def evaluate_results(labels, preds, probs, split_name):
    """Compute and display full evaluation metrics."""
    acc = accuracy_score(labels, preds)
    report = classification_report(labels, preds, target_names=LABEL_NAMES,
                                    digits=4, output_dict=True)
    report_str = classification_report(labels, preds, target_names=LABEL_NAMES, digits=4)

    print(f"\n{'='*60}")
    print(f"  {split_name.upper()} SET RESULTS")
    print(f"{'='*60}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"\n{report_str}")

    return {
        "predictions": preds,
        "labels": labels,
        "probabilities": probs,
        "accuracy": acc,
        "report": report,
    }

dev_labels = dev_df["label"].values
test_labels = test_df["label"].values

dev_results = evaluate_results(dev_labels, dev_preds, dev_probs, "Dev")
test_results = evaluate_results(test_labels, test_preds, test_probs, "Test")